# Model Evaluation: NJ Housing Price Predictor

Evaluates the QLoRA fine-tuned Qwen2.5-0.5B on held-out test data.

## What this notebook does
1. Mounts Google Drive and loads the trained LoRA adapter
2. Loads test data from HuggingFace dataset (rajkumar4466/nj-housing-prices)
3. Runs inference on test set (200-sample quick check, then full set)
4. Computes regression metrics: MAE, RMSE, R-squared, MAPE
5. Generates predicted vs actual scatter plot
6. Regenerates training loss curve from saved log history

## Requirements
- Google Colab with GPU runtime (T4 or A100)
- Trained LoRA adapter on Google Drive at housing_model/lora_adapter/
- trainer_log_history.json on Google Drive (saved by 02_train.ipynb)
- HuggingFace dataset: rajkumar4466/nj-housing-prices

In [ ]:
import os
import time

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/housing_model"
DRIVE_ADAPTER = os.path.join(DRIVE_BASE, "lora_adapter")
DRIVE_PLOTS = os.path.join(DRIVE_BASE, "plots")

os.makedirs(DRIVE_PLOTS, exist_ok=True)

# Verify adapter exists
assert os.path.isdir(DRIVE_ADAPTER), f"Adapter not found at {DRIVE_ADAPTER}. Run 02_train.ipynb first."
assert os.path.isfile(os.path.join(DRIVE_ADAPTER, "adapter_config.json")), "adapter_config.json missing"

print(f"Drive paths configured:")
print(f"  Adapter: {DRIVE_ADAPTER}")
print(f"  Plots:   {DRIVE_PLOTS}")
print(f"Adapter found on Drive.")

_start_time = time.time()

In [ ]:
%%capture install_output
!pip install \
    transformers==5.2.0 \
    peft==0.18.1 \
    bitsandbytes==0.49.2 \
    accelerate==1.12.0 \
    datasets==4.6.0 \
    sentencepiece==0.2.1 \
    scikit-learn==1.8.0 \
    matplotlib==3.10.8

In [ ]:
print("Dependencies installed. Key packages:")
import transformers, peft, bitsandbytes, accelerate
print(f"  transformers: {transformers.__version__}")
print(f"  peft:         {peft.__version__}")
print(f"  bitsandbytes: {bitsandbytes.__version__}")
print(f"  accelerate:   {accelerate.__version__}")

In [ ]:
import json
import importlib
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datasets import load_dataset

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}")

In [ ]:
HF_DATASET = "rajkumar4466/nj-housing-prices"
print(f"Loading dataset from HuggingFace: {HF_DATASET}...")

ds = load_dataset(HF_DATASET)
test_data = ds["test"]

print(f"Test set: {len(test_data):,} records")
print(f"Columns: {test_data.column_names}")
print(f"\nSample test record:")
print(f"  Prompt: {test_data[0]['prompt'][:100]}...")
print(f"  Price:  ${test_data[0]['price']:,.0f}")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"

# Load in 4-bit (same config as training) -- saves VRAM for inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} with 4-bit quantization...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading LoRA adapter from {DRIVE_ADAPTER}...")
model = PeftModel.from_pretrained(base_model, DRIVE_ADAPTER)
model.eval()
print(f"Model loaded with adapter. Ready for inference.")

In [ ]:
# Clone repo or ensure lambda/prompt_utils.py is accessible
# Option 1: If repo is cloned on Colab
# Option 2: Define inline as fallback

import re

def parse_price_from_output(generated_text: str) -> float | None:
    """
    Extract the first numeric price from model-generated text.
    Mirrors lambda/prompt_utils.py -- kept inline for Colab portability.
    """
    cleaned = generated_text.replace(",", "")
    match = re.search(r"\d+(?:\.\d+)?", cleaned)
    if match:
        return float(match.group())
    return None

# Quick test
assert parse_price_from_output("425000") == 425000.0
assert parse_price_from_output("$1,250,000") == 1250000.0
assert parse_price_from_output("no number here") is None
print("parse_price_from_output loaded and tested.")

In [ ]:
def predict_price(prompt: str) -> tuple[float | None, str]:
    """
    Run inference on a single prompt.
    Returns (parsed_price, raw_generated_text).
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            temperature=1.0,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the part after the prompt
    generated_suffix = generated[len(prompt):]
    parsed = parse_price_from_output(generated_suffix)
    return parsed, generated_suffix


# Quick smoke test on first record
test_price, test_raw = predict_price(test_data[0]["prompt"])
print(f"Smoke test:")
print(f"  Raw output: {test_raw[:50]}")
print(f"  Parsed: {test_price}")
print(f"  Actual: ${test_data[0]['price']:,.0f}")

In [ ]:
# Run on 200-sample subset first for quick metric check
QUICK_N = 200
quick_indices = list(range(min(QUICK_N, len(test_data))))

print(f"Running quick evaluation on {len(quick_indices)} samples...")
quick_start = time.time()

quick_actual = []
quick_predicted = []
quick_parse_failures = 0

for i, idx in enumerate(quick_indices):
    record = test_data[idx]
    pred, raw = predict_price(record["prompt"])
    if pred is not None:
        quick_actual.append(record["price"])
        quick_predicted.append(pred)
    else:
        quick_parse_failures += 1
    if (i + 1) % 50 == 0:
        elapsed = time.time() - quick_start
        rate = (i + 1) / elapsed
        print(f"  {i+1}/{len(quick_indices)} done ({rate:.1f} samples/sec)")

quick_elapsed = time.time() - quick_start
print(f"\nQuick eval complete in {quick_elapsed:.0f}s ({len(quick_indices)/quick_elapsed:.1f} samples/sec)")
print(f"Parse failures: {quick_parse_failures}/{len(quick_indices)} ({100*quick_parse_failures/len(quick_indices):.1f}%)")

if quick_parse_failures / len(quick_indices) > 0.05:
    print("WARNING: Parse failure rate exceeds 5%. Model may not be generating valid prices.")

# Quick metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

quick_mae = mean_absolute_error(quick_actual, quick_predicted)
quick_rmse = np.sqrt(mean_squared_error(quick_actual, quick_predicted))
quick_r2 = r2_score(quick_actual, quick_predicted)
quick_mape = np.mean(np.abs((np.array(quick_actual) - np.array(quick_predicted)) / np.array(quick_actual))) * 100

print(f"\nQuick Metrics (n={len(quick_actual)}):")
print(f"  MAE:  ${quick_mae:,.0f}")
print(f"  RMSE: ${quick_rmse:,.0f}")
print(f"  R2:   {quick_r2:.4f}")
print(f"  MAPE: {quick_mape:.1f}%")

# Estimate full eval time
est_full_time = quick_elapsed / len(quick_indices) * len(test_data)
print(f"\nEstimated full eval time: {est_full_time/60:.1f} min")

In [ ]:
# Full test set evaluation
print(f"Running full evaluation on {len(test_data)} samples...")
full_start = time.time()

y_actual = []
y_predicted = []
parse_failures = 0
failure_examples = []

for i in range(len(test_data)):
    record = test_data[i]
    pred, raw = predict_price(record["prompt"])
    if pred is not None:
        y_actual.append(record["price"])
        y_predicted.append(pred)
    else:
        parse_failures += 1
        if len(failure_examples) < 5:
            failure_examples.append({"idx": i, "raw": raw[:80], "prompt_end": record["prompt"][-40:]})
    if (i + 1) % 100 == 0:
        elapsed = time.time() - full_start
        rate = (i + 1) / elapsed
        remaining = (len(test_data) - i - 1) / rate
        print(f"  {i+1}/{len(test_data)} done ({rate:.1f}/sec, ~{remaining/60:.1f} min remaining)")

full_elapsed = time.time() - full_start
print(f"\nFull evaluation complete in {full_elapsed/60:.1f} minutes")
print(f"Valid predictions: {len(y_actual)}/{len(test_data)}")
print(f"Parse failures: {parse_failures}/{len(test_data)} ({100*parse_failures/len(test_data):.1f}%)")

if parse_failures / len(test_data) > 0.05:
    print("\nWARNING: Parse failure rate exceeds 5%!")
    print("Failure examples:")
    for ex in failure_examples:
        print(f"  idx={ex['idx']}: raw='{ex['raw']}' prompt_end='{ex['prompt_end']}'")

In [ ]:
# EVAL-01: Compute all 4 regression metrics on full test set
mae = mean_absolute_error(y_actual, y_predicted)
rmse = np.sqrt(mean_squared_error(y_actual, y_predicted))
r2 = r2_score(y_actual, y_predicted)
mape = np.mean(np.abs((np.array(y_actual) - np.array(y_predicted)) / np.array(y_actual))) * 100

print("=" * 60)
print("EVALUATION METRICS (EVAL-01)")
print("=" * 60)
print(f"Test set size:    {len(test_data):,} records")
print(f"Valid predictions: {len(y_actual):,} records")
print(f"Parse failures:   {parse_failures:,} records ({100*parse_failures/len(test_data):.1f}%)")
print(f"")
print(f"MAE:  ${mae:,.0f}")
print(f"RMSE: ${rmse:,.0f}")
print(f"R2:   {r2:.4f}")
print(f"MAPE: {mape:.1f}%")
print("=" * 60)

# Save metrics to JSON on Drive for reference
metrics = {
    "test_set_size": len(test_data),
    "valid_predictions": len(y_actual),
    "parse_failures": parse_failures,
    "parse_failure_rate": parse_failures / len(test_data),
    "mae": float(mae),
    "rmse": float(rmse),
    "r2": float(r2),
    "mape": float(mape),
    "eval_time_seconds": full_elapsed,
}
metrics_path = os.path.join(DRIVE_BASE, "eval_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to: {metrics_path}")

In [ ]:
# EVAL-02: Predicted vs actual scatter plot
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_actual, y_predicted, alpha=0.3, s=10, color="steelblue", label="Predictions")

# y=x reference line (perfect prediction)
lims = [
    min(min(y_actual), min(y_predicted)),
    max(max(y_actual), max(y_predicted)),
]
ax.plot(lims, lims, "r--", label="Perfect prediction", linewidth=1.5)

ax.set_xlabel("Actual Price ($)", fontsize=12)
ax.set_ylabel("Predicted Price ($)", fontsize=12)
ax.set_title("Predicted vs. Actual Housing Prices\nQwen2.5-0.5B QLoRA Fine-tuned", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Format axes as currency
from matplotlib.ticker import FuncFormatter
currency_fmt = FuncFormatter(lambda x, p: f"${x:,.0f}")
ax.xaxis.set_major_formatter(currency_fmt)
ax.yaxis.set_major_formatter(currency_fmt)

plt.tight_layout()
scatter_path = os.path.join(DRIVE_PLOTS, "predicted_vs_actual.png")
plt.savefig(scatter_path, dpi=150)
plt.show()
plt.close()

print(f"Scatter plot saved to: {scatter_path}")

In [ ]:
# EVAL-03: Training loss curve regenerated from trainer log history
log_history_path = os.path.join(DRIVE_BASE, "trainer_log_history.json")

if not os.path.isfile(log_history_path):
    print(f"WARNING: {log_history_path} not found.")
    print("The training notebook (02_train.ipynb) must save trainer.state.log_history as JSON.")
    print("Falling back to displaying existing loss curve PNG if available.")
    existing_png = os.path.join(DRIVE_PLOTS, "training_loss_curve.png")
    if os.path.isfile(existing_png):
        from IPython.display import Image, display
        display(Image(filename=existing_png))
        print(f"Displayed existing loss curve from: {existing_png}")
    else:
        print("No loss curve available. Skipping EVAL-03.")
else:
    with open(log_history_path) as f:
        log_history = json.load(f)

    train_losses = [(entry["step"], entry["loss"]) for entry in log_history if "loss" in entry]
    eval_losses = [(entry["step"], entry["eval_loss"]) for entry in log_history if "eval_loss" in entry]

    fig, ax = plt.subplots(figsize=(10, 5))

    if train_losses:
        steps, losses = zip(*train_losses)
        ax.plot(steps, losses, label="Training Loss", color="steelblue", alpha=0.8)

    if eval_losses:
        eval_steps_vals, eval_loss_vals = zip(*eval_losses)
        ax.plot(eval_steps_vals, eval_loss_vals, label="Validation Loss", color="coral",
                alpha=0.8, marker="o", markersize=4)

    ax.set_xlabel("Training Step", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("QLoRA Training Loss -- Qwen2.5-0.5B on NJ Housing Data", fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    if train_losses:
        final_step, final_loss = train_losses[-1]
        ax.annotate(f"Final: {final_loss:.4f}", xy=(final_step, final_loss),
                    xytext=(-60, 20), textcoords="offset points",
                    arrowprops=dict(arrowstyle="->", color="gray"),
                    fontsize=9, color="steelblue")

    plt.tight_layout()
    loss_curve_path = os.path.join(DRIVE_PLOTS, "training_loss_curve.png")
    plt.savefig(loss_curve_path, dpi=150)
    plt.show()
    plt.close()

    print(f"Training loss curve saved to: {loss_curve_path}")
    if train_losses:
        first_loss = train_losses[0][1]
        last_loss = train_losses[-1][1]
        print(f"Loss trajectory: {first_loss:.4f} -> {last_loss:.4f} ({'decreased' if last_loss < first_loss else 'DID NOT decrease'})")

In [ ]:
total_elapsed = time.time() - _start_time

print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"Model:            Qwen/Qwen2.5-0.5B + QLoRA adapter")
print(f"Test set:         {len(test_data):,} records")
print(f"Valid predictions: {len(y_actual):,}")
print(f"Parse failures:   {parse_failures:,} ({100*parse_failures/len(test_data):.1f}%)")
print(f"")
print(f"MAE:  ${mae:,.0f}")
print(f"RMSE: ${rmse:,.0f}")
print(f"R2:   {r2:.4f}")
print(f"MAPE: {mape:.1f}%")
print(f"")
print(f"Plots saved to: {DRIVE_PLOTS}")
print(f"  - predicted_vs_actual.png")
print(f"  - training_loss_curve.png")
print(f"Metrics saved to: {os.path.join(DRIVE_BASE, 'eval_metrics.json')}")
print(f"")
print(f"Total evaluation time: {total_elapsed/60:.1f} minutes")
print("=" * 60)

# Requirement checks
print("\nRequirement Status:")
print(f"  EVAL-01 (metrics): {'PASSED' if len(y_actual) > 0 else 'FAILED'} -- {len(y_actual)} valid predictions, all 4 metrics computed")
scatter_exists = os.path.isfile(os.path.join(DRIVE_PLOTS, "predicted_vs_actual.png"))
print(f"  EVAL-02 (scatter): {'PASSED' if scatter_exists else 'FAILED'}")
loss_exists = os.path.isfile(os.path.join(DRIVE_PLOTS, "training_loss_curve.png"))
print(f"  EVAL-03 (loss curve): {'PASSED' if loss_exists else 'FAILED'}")